# Template Quest Notebook

In [1]:
# Import neccessary modules, add to this cell as needed
# Provided PySpark examples

from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.functions import (
    col, get_json_object, when, count, coalesce,
    isnan, isnull, upper, lower, trim, explode_outer, from_json
)
from pyspark.sql.types import (
    IntegerType, LongType, StringType, TimestampType, 
    StructType, StructField, DoubleType
)
import json
import pandas as pd

print(" All imports successful")

 All imports successful


## Part 1: Load the Sample Dataset

In [2]:
# Initiate a new Spark session and set the case sensitivity option
spark = (
    SparkSession.builder
        .appName("cyberquest")
        .getOrCreate()
)
spark.conf.set("spark.sql.caseSensitive", True)

print(f" Spark {spark.version} initialized")
print(f"   App: cyberquest")
print(f"   Case Sensitive: True")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/26 01:35:30 WARN Utils: Your hostname, kali, resolves to a loopback address: 127.0.1.1; using 192.168.30.130 instead (on interface eth0)
26/08/26 01:35:30 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/kali/cyber-quest/venv/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/08/26 01:35:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


 Spark 4.2.0 initialized
   App: cyberquest
   Case Sensitive: True


In [3]:
# TODO: Load the raw data
print("="*80)
print("PART 1: LOAD THE SAMPLE DATASET (BRONZE LAYER)")
print("="*80)

print("\nStep 1: Loading JSON line file from data/ directory...")

# Load all JSON files from the data directory
df_bronze = spark.read.json("data/sysmon_spearphish_cribl.json" )

print(f"\n BRONZE layer created successfully")
print(f"   Total rows: {df_bronze.count():,}")
print(f"   Total columns: {len(df_bronze.columns)}")

print(f"\nBRONZE Schema:")
df_bronze.printSchema()

print(f"\nBRONZE Sample Data (first row):")
df_bronze.limit(1).show(vertical=True, truncate=False)

PART 1: LOAD THE SAMPLE DATASET (BRONZE LAYER)

Step 1: Loading JSON line file from data/ directory...



 BRONZE layer created successfully


   Total rows: 28,017
   Total columns: 10

BRONZE Schema:
root
 |-- Computer: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- EventCode: string (nullable = true)
 |-- User: string (nullable = true)
 |-- _raw: string (nullable = true)
 |-- _time: double (nullable = true)
 |-- cribl_breaker: string (nullable = true)
 |-- cribl_pipe: string (nullable = true)
 |-- host: string (nullable = true)
 |-- source: string (nullable = true)


BRONZE Sample Data (first 15 row to understand log fields):


-RECORD 0-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [4]:
# TODO: Parse the raw data into something relevant and usable
print("\n" + "="*80)
print("STEP 2: PARSE JSON FROM _RAW (SILVER LAYER)")
print("="*80)

print("\nParsing JSON fields from _raw column...\n")

# Extract relevant fields for threat hunting
# Using the medallion architecture: Bronze -> Silver
df_silver = df_bronze.select(
    # === METADATA FROM OUTER COLUMNS ===
    col("_time").alias("event_time"),
    col("host").alias("log_host"),
    col("source").alias("log_source"),
    col("Computer").alias("computer"),
    col("user").alias("user"),
    col("EventCode").alias("eventcode"),

     # === EVENT INFORMATION (from _raw JSON) ===
    get_json_object(col("_raw"), "$.EventType").alias("event_type"),
    get_json_object(col("_raw"), "$.Task").alias("event_id"),
    
    #====time=====#
    get_json_object(col("_raw"), "$.UtcTime").alias("utc_time"),
    
    
    # === PROCESS EXECUTION (for detecting Office apps) ===
    get_json_object(col("_raw"), "$.Image").alias("image"),
    get_json_object(col("_raw"), "$.CommandLine").alias("command_line"),
    get_json_object(col("_raw"), "$.ProcessId").alias("process_id"),
    get_json_object(col("_raw"), "$.ParentImage").alias("parent_image"),
    get_json_object(col("_raw"), "$.ParentProcessId").alias("parent_process_id"),
    get_json_object(col("_raw"), "$.ParentCommandLine").alias("parent_command_line"),
    get_json_object(col("_raw"), "$.ProcessGuid").alias("process_guid"),
    get_json_object(col("_raw"), "$.ParentProcessGuid").alias("parent_process_guid"),
    get_json_object(col("_raw"), "$.Hashes").alias("hashes"),
    
    
    # === DNS QUERIES (for detecting malicious domains) ===
    get_json_object(col("_raw"), "$.QueryName").alias("query_name"),
    get_json_object(col("_raw"), "$.QueryStatus").alias("query_status"),
    get_json_object(col("_raw"), "$.QueryResults").alias("query_results"),
    
    # === NETWORK INFORMATION ===
    get_json_object(col("_raw"), "$.SourceIp").alias("source_ip"),
    get_json_object(col("_raw"), "$.DestinationIp").alias("destination_ip"),
    get_json_object(col("_raw"), "$.SourcePort").alias("source_port"),
    get_json_object(col("_raw"), "$.DestinationPort").alias("destination_port"),
    
    # Keep raw for reference
    col("_raw").alias("raw_event")
)

print(f" SILVER layer created successfully")
print(f"   Total columns: {len(df_silver.columns)}")

# Convert data types
print("\nConverting data types...")
df_silver = df_silver \
    .withColumn("process_id", col("process_id").cast(IntegerType())) \
    .withColumn("parent_process_id", col("parent_process_id").cast(IntegerType())) \
    .withColumn("query_status", col("query_status").cast(IntegerType())) \
    .withColumn("source_port", col("source_port").cast(IntegerType())) \
    .withColumn("destination_port", col("destination_port").cast(IntegerType())) \
    .withColumn("event_time", col("event_time").cast(TimestampType()))

print(" Data types converted\n")

print("SILVER Schema:")
df_silver.printSchema()

# Create the SQL view as required
df_silver.createOrReplaceTempView("sysmon_silver")

print(f"\n Temporary view 'sysmon_silver' created")


STEP 2: PARSE JSON FROM _RAW (SILVER LAYER)

Parsing JSON fields from _raw column...

 SILVER layer created successfully
   Total columns: 26

Converting data types...
 Data types converted

SILVER Schema:
root
 |-- event_time: timestamp (nullable = true)
 |-- log_host: string (nullable = true)
 |-- log_source: string (nullable = true)
 |-- computer: string (nullable = true)
 |-- user: string (nullable = false)
 |-- eventcode: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- utc_time: string (nullable = true)
 |-- image: string (nullable = true)
 |-- command_line: string (nullable = true)
 |-- process_id: integer (nullable = true)
 |-- parent_image: string (nullable = true)
 |-- parent_process_id: integer (nullable = true)
 |-- parent_command_line: string (nullable = true)
 |-- process_guid: string (nullable = true)
 |-- parent_process_guid: string (nullable = true)
 |-- hashes: string (nullable = true)
 |-- query_name: st

26/08/26 01:38:32 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.



 Temporary view 'sysmon_silver' created


In [5]:
# Provided PySpark Example
df_silver.limit(5).show()

+--------------------+------------+--------------------+--------------------+----+---------+----------+--------+--------------------+--------------------+------------+----------+------------+-----------------+-------------------+--------------------+-------------------+--------------------+----------+------------+-------------+---------+--------------+-----------+----------------+--------------------+
|          event_time|    log_host|          log_source|            computer|user|eventcode|event_type|event_id|            utc_time|               image|command_line|process_id|parent_image|parent_process_id|parent_command_line|        process_guid|parent_process_guid|              hashes|query_name|query_status|query_results|source_ip|destination_ip|source_port|destination_port|           raw_event|
+--------------------+------------+--------------------+--------------------+----+---------+----------+--------+--------------------+--------------------+------------+----------+------------

In [6]:
# Provided PySpark SQL Example
spark.sql("""
SELECT *
FROM sysmon_silver LIMIT 5
""").show()

+--------------------+------------+--------------------+--------------------+----+---------+----------+--------+--------------------+--------------------+------------+----------+------------+-----------------+-------------------+--------------------+-------------------+--------------------+----------+------------+-------------+---------+--------------+-----------+----------------+--------------------+
|          event_time|    log_host|          log_source|            computer|user|eventcode|event_type|event_id|            utc_time|               image|command_line|process_id|parent_image|parent_process_id|parent_command_line|        process_guid|parent_process_guid|              hashes|query_name|query_status|query_results|source_ip|destination_ip|source_port|destination_port|           raw_event|
+--------------------+------------+--------------------+--------------------+----+---------+----------+--------+--------------------+--------------------+------------+----------+------------

Part 2: Detection Engineering

In [7]:
#Find DNS queries from those Office processes, excluding known-good MS domains
# Sysmon eventcode 22, This event logs every time a process requests a domain name resolution, regardless of whether the query succeeds or fails. 
spark.sql("""
SELECT computer, user, utc_time, image, query_name, query_status
FROM sysmon_silver
WHERE eventcode = '22'
  AND image RLIKE 'WINWORD.EXE|EXCEL.EXE|POWERPNT.EXE|OUTLOOK.EXE|ONENOTE.EXE'
  AND query_name NOT RLIKE '\\.office\\.com$|\\.office\\.net$|\\.microsoft\\.com$|\\.skype\\.com$|\\.msedge\\.net$|\\.trafficmanager\\.net$|sfx\\.ms$'
""").show(truncate=False)


[Stage 8:>                                                          (0 + 1) / 1]

+------------------------------+----+-----------------------+-----------------------------------------------------------+-----------------+------------+
|computer                      |user|utc_time               |image                                                      |query_name       |query_status|
+------------------------------+----+-----------------------+-----------------------------------------------------------+-----------------+------------+
|win-host-ctus-attack-range-212|kali|2023-01-27 11:30:14.523|C:\Program Files\Microsoft Office\root\Office16\WINWORD.EXE|www.mediafire.com|0           |
+------------------------------+----+-----------------------+-----------------------------------------------------------+-----------------+------------+



In [8]:
### this query is correlaing event_id 1 (process creation) with event_id 22(DNS request). After correlation we can get more details about process creation events like parent process commandline etc

spark.sql("""
WITH office_launch AS (
  SELECT computer, user, process_guid, process_id, utc_time AS launch_time, 
         command_line, parent_image
  FROM sysmon_silver
  WHERE eventcode = '1' AND image RLIKE 'WINWORD.EXE|EXCEL.EXE|POWERPNT.EXE|OUTLOOK.EXE|ONENOTE.EXE'
),
suspicious_dns AS (
  SELECT computer, process_guid, utc_time AS dns_time, query_name
  FROM sysmon_silver
  WHERE eventcode = '22'
    AND query_name NOT RLIKE '\\.office\\.com$|\\.office\\.net$|\\.microsoft\\.com$|\\.skype\\.com$|\\.msedge\\.net$|\\.trafficmanager\\.net$|sfx\\.ms$'
)
SELECT o.computer, o.user, o.launch_time, o.command_line, o.parent_image,
       d.dns_time, d.query_name
FROM office_launch o
JOIN suspicious_dns d ON o.computer = d.computer AND o.process_guid = d.process_guid
ORDER BY o.launch_time
""").show(truncate=False)

[Stage 13:>                                                         (0 + 1) / 1]

+------------------------------+----+-----------------------+-------------------------------------------------------------------------------------------------------------+-----------------------+-----------------------+-----------------+
|computer                      |user|launch_time            |command_line                                                                                                 |parent_image           |dns_time               |query_name       |
+------------------------------+----+-----------------------+-------------------------------------------------------------------------------------------------------------+-----------------------+-----------------------+-----------------+
|win-host-ctus-attack-range-212|kali|2023-01-27 11:30:11.855|"C:\Program Files\Microsoft Office\Root\Office16\WINWORD.EXE" /n "C:\Temp\asyncrat\loader\asyncrat.doc" /o ""|C:\Windows\explorer.exe|2023-01-27 11:30:14.523|www.mediafire.com|
+------------------------------+----+-----------

## Part 3: Additional Steps

### Part 3.1: Normalization

In [9]:
### Normalize to MITRE ATT&CK + OCSF-style schema
detection_df = spark.sql("""
WITH office_launch AS (
  SELECT computer, user, process_guid, parent_process_guid, utc_time AS launch_time, 
         command_line, image AS process_image, parent_image
  FROM sysmon_silver
  WHERE eventcode = '1' AND image RLIKE 'WINWORD.EXE|EXCEL.EXE|POWERPNT.EXE|OUTLOOK.EXE|ONENOTE.EXE'
),
suspicious_dns AS (
  SELECT computer, process_guid, utc_time AS dns_time, query_name
  FROM sysmon_silver
  WHERE eventcode = '22'
    AND query_name NOT RLIKE '\\.office\\.com$|\\.office\\.net$|\\.microsoft\\.com$|\\.skype\\.com$|\\.msedge\\.net$|\\.trafficmanager\\.net$|sfx\\.ms$'
)
SELECT 
    -- Normalized/OCSF-style fields
    o.computer                     AS device_name,
    o.user                         AS actor_user,
    o.launch_time                  AS event_time,
    o.parent_image                 AS process_parent_name,
    o.process_image                AS process_name,
    o.command_line                 AS process_cmdline,
    d.query_name                   AS dns_query,
    'T1566.001'                    AS mitre_technique,   -- Spearphishing Attachment
    'T1071.004'                    AS mitre_technique_2, -- App Layer Protocol: DNS
    'Initial Access'               AS mitre_tactic
FROM office_launch o
JOIN suspicious_dns d 
  ON o.computer = d.computer AND o.process_guid = d.process_guid
""")

detection_df.show(truncate=False)

[Stage 15:=============================>                            (1 + 1) / 2]

+------------------------------+----------+-----------------------+-----------------------+-----------------------------------------------------------+-------------------------------------------------------------------------------------------------------------+-----------------+---------------+-----------------+--------------+
|device_name                   |actor_user|event_time             |process_parent_name    |process_name                                               |process_cmdline                                                                                              |dns_query        |mitre_technique|mitre_technique_2|mitre_tactic  |
+------------------------------+----------+-----------------------+-----------------------+-----------------------------------------------------------+-------------------------------------------------------------------------------------------------------------+-----------------+---------------+-----------------+--------------+
|win-host-ctu

### Part 3.2: Alert Table

In [10]:
from pyspark.sql.functions import col, lit, when, sha2, concat, current_timestamp

alert_table = detection_df.select(

    # --- Alert identity ---
    sha2(concat(col("device_name"), col("process_cmdline"), col("event_time").cast("string")), 256).alias("alert_id"),
    lit("Suspicious Office Application DNS Activity").alias("signature"),
    lit("PySql-OFFICE-DNS-001").alias("rule_id"),

    # --- Timing ---
    col("event_time"),
    current_timestamp().alias("detected_at"),

    # --- Severity (derived here since detection_df doesn't have it) ---
    when(col("dns_query").rlike("mediafire|dropbox|mega\\.nz|pastebin"), lit("High"))
        .otherwise(lit("Medium")).alias("severity"),
    lit("High").alias("confidence"),

    # --- Asset / identity context ---
    col("device_name"),
    col("actor_user"),

    # --- Process context ---
    col("process_parent_name"),
    col("process_name"),
    col("process_cmdline"),

    # --- DNS context ---
    col("dns_query"),

    # --- ATT&CK mapping (already in detection_df) ---
    col("mitre_technique"),
    col("mitre_technique_2"),
    col("mitre_tactic"),

    # --- Analyst-facing summary ---
    concat(
        lit("Office process "), col("process_name"),
        lit(" was launched by "), col("process_parent_name"),
        lit(" with suspicious command line ["), col("process_cmdline"),
        lit("] and subsequently queried non-Microsoft domain "), col("dns_query"),
        lit(". Consistent with a malicious document delivering a staged payload.")
    ).alias("alert_description"),

    lit("New").alias("status")
)

alert_table.show(truncate=False)

alert_table.createOrReplaceTempView("analyst_alerts")

+----------------------------------------------------------------+------------------------------------------+--------------------+-----------------------+--------------------------+--------+----------+------------------------------+----------+-----------------------+-----------------------------------------------------------+-------------------------------------------------------------------------------------------------------------+-----------------+---------------+-----------------+--------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------+
|alert_id                                                        |signature                          

### Part 3.3: Enrichment

In [22]:
import requests
import time

VT_API_KEY = "api"  

def check_virustotal(domain):
    """Query VirusTotal for domain reputation, category, and detections"""
    try:
        resp = requests.get(
            f"https://www.virustotal.com/api/v3/domains/{domain}",
            headers={"x-apikey": VT_API_KEY},
            timeout=10
        )
        print(resp.status_code)
        attrs = resp.json().get("data", {}).get("attributes", {})
        stats = attrs.get("last_analysis_stats", {})
        categories = attrs.get("categories", {})

        return {
            "domain": domain,
            "malicious_votes": stats.get("malicious", 0),
            "suspicious_votes": stats.get("suspicious", 0),
            "harmless_votes": stats.get("harmless", 0),
            "reputation_score": attrs.get("reputation"),
            "category": ", ".join(set(categories.values())) if categories else "uncategorized",
            "domain_created": attrs.get("creation_date"),  # unix timestamp
            "registrar": attrs.get("registrar")
        }
    except Exception as e:
        return {"domain": domain, "error": str(e)}

# Enrich each unique domain found in the alerts
domains = [row["dns_query"] for row in detection_df.select("dns_query").distinct().collect()]
enrichment_results = []
for d in domains:
    result = check_virustotal(d)
    enrichment_results.append(result)
    time.sleep(1)

import pandas as pd
enrichment_pd = pd.DataFrame(enrichment_results)
enrichment_pd["domain_created"] = pd.to_datetime(enrichment_pd["domain_created"], unit="s", errors="coerce")
print(enrichment_pd.to_string(index=False))

200
           domain  malicious_votes  suspicious_votes  harmless_votes  reputation_score      category      domain_created        registrar
www.mediafire.com                2                 0              59                62 uncategorized 2002-08-11 15:31:16 Cloudflare, Inc.


## Summary

Summarize your submission here, comments are helpful to add throughout your code as well.


## Summary

**Hypothesis validated:** A spearphishing document delivered via Office led to malicious 
DNS activity, consistent with a maldoc-to-RAT delivery chain.

**Finding:** Out of 28,017 Sysmon events, correlating Office process launches (EventCode 1) 
with their subsequent DNS queries (EventCode 22) — excluding known Microsoft domains — 
surfaced exactly one high-fidelity detection:

`WINWORD.EXE` (parent: `explorer.exe`) launched with command line referencing 
`C:\Temp\asyncrat\loader\asyncrat.doc`, then queried `www.mediafire.com`.

**Correlation:** Joined on `ProcessGuid` and `computer`

**Enrichment:** VirusTotal lookup on `www.mediafire.com` returned 2 malicious / 0 suspicious 
/ 59 harmless vendor votes, reputation score 62, and a creation date of 2002-08-11 
(registrar: Cloudflare, Inc.) — a 20+ year old, largely trusted domain. Reputation/age alone 
would not have flagged this activity; the detection signal came entirely from process and 
command-line context, with the 2 malicious votes serving as weak corroborating evidence 
that mediafire has been abused for payload staging before.

**Limitations:** Single dataset, single detection instance — DNS allowlist and severity 
thresholds would need tuning against a larger, more diverse traffic baseline before 
production use.

**AI usage:** Claude was used to scaffold PySpark/SQL and validate field names against the 
actual JSON schema.